In [2]:
import numpy as np

import xrt.backends.raycing as raycing
import xrt.backends.raycing.sources as sources
import xrt.backends.raycing.oes as oes
import xrt.runner as runner


# Geometry: source -> mirror = 12 m, mirror -> focus = 6 m
# Therefore, source-to-focus = 18 m and q / p = 1 / 2.
P = 12.0
Q = 6.0
grazing_angle = 3e-3  # radians

# Toroidal mirror radii for tangential and sagittal focusing.
# 1/P + 1/Q = 2 sin(theta) / Rm
# 1/P + 1/Q = 2 / (Rs sin(theta))
Rm = 2.0 * P * Q / ((P + Q) * np.sin(grazing_angle))
Rs = 2.0 * P * Q * np.sin(grazing_angle) / (P + Q)

beamline = raycing.BeamLine()

source = sources.GeometricSource(
    beamline,
    name="Gaussian source",
    center=(0.0, 0.0, 0.0),
    nrays=100000,
    distx="normal",
    dx=50e-6,
    distz="normal",
    dz=50e-6,
    distxprime="normal",
    dxprime=0.1e-3,
    distzprime="normal",
    dzprime=0.1e-3,
    energies=(8000.0,),
    polarization="horizontal",
)

mirror = oes.OE(
    beamline,
    name="Toroidal mirror",
    center=(0.0, P, 0.0),
    surface="toroid",
    Rm=Rm,
    Rs=Rs,
    pitch=np.pi / 2.0 - grazing_angle,
    limPhysX=(-0.05, 0.05),
    limPhysY=(-0.10, 0.10),
)


def run_process(beamline):
    source_beam = beamline.source.shine()
    reflected_beam, local_beam = beamline.mirror.reflect(source_beam)

    return {
        "source": source_beam,
        "mirror": reflected_beam,
        "mirror_local": local_beam,
    }


runner.run_process = run_process

print(f"Mirror position: {P:.3f} m")
print(f"Focus position:  {P + Q:.3f} m")
print(f"Tangential radius Rm: {Rm:.6g} m")
print(f"Sagittal radius Rs:   {Rs:.6g} m")

runner.run_ray_tracing(
    beamLine=beamline,
    repeats=1,
    nrays=100000,
)

Mirror position: 12.000 m
Focus position:  18.000 m
Tangential radius Rm: 2666.67 m
Sagittal radius Rs:   0.024 m


TypeError: run_ray_tracing() got an unexpected keyword argument 'nrays'